<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [31]:
import torch
import numpy as np
import pandas as pd
import albumentations as A

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from albumentations.pytorch.transforms import ToTensorV2

In [32]:
import torch.nn as nn
import torch.nn.functional as F
import io
from torchvision.ops import box_iou, nms, distance_box_iou_loss
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models.detection.anchor_utils import AnchorGenerator

### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [33]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

Создаем датасет для предобработки данных

In [34]:
class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Загружаем данные и разметку для объекта с индексом `idx`.

        labels: List[int] Набор классов для каждого ббокса,
        boxes: List[List[int]] Набор ббоксов в формате (x_min, y_min, w, h).
        """
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        # Вычитаем единицу чтобы классы начинались с нуля
        labels = [label - 1 for label in labels]
        boxes = row['bbox'].tolist()

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        target['boxes'] = torch.tensor(np.array(boxes), dtype=torch.float32)
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

def collate_fn(batch):
    batch = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [35]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        # Добавляй сюда свои аугментации при необходимости!
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    # Раскомментируй, если аугментации изменяют ббоксы.
    # Не забудь указать верный формат для ббоксов.
    # bbox_params=A.BboxParams(format='coco', label_fields=['labels'])
)

test_transform = A.Compose(
    [
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ]
)

Не забываем инициализировать наш датасет

In [36]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [37]:
class Backbone(nn.Module):
    def __init__(self, unfreeze_last=0):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.DEFAULT)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        for param in self.parameters():
            param.requires_grad = False

        self.blocks = [self.stem, self.layer1, self.layer2, self.layer3, self.layer4]
        for block in self.blocks[-unfreeze_last:]:
            for param in block.parameters():
                param.requires_grad = True

        self.out_channels = [256, 512, 1024, 2048]

    def forward(self, x):
        x = self.stem(x)
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return c2, c3, c4

### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [38]:
import torch.nn as nn

class Neck(nn.Module):
    def __init__(self, in_channels_list, out_channels=256 ):
        super().__init__()
        self.out_channels = out_channels

        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            for in_channels in in_channels_list
        ])

        self.fpn_convs = nn.ModuleList([
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
            for _ in in_channels_list
        ])

    def forward(self, features):
        laterals = []

        for i, (feature, lateral_conv) in enumerate(
            zip(reversed(features), reversed(self.lateral_convs))
        ):
            if i == 0:
                lateral = lateral_conv(feature)
            else:
                prev_lateral = laterals[-1]
                upsampled = F.interpolate(
                    prev_lateral,
                    scale_factor=2.0,
                    mode='nearest'
                )
                lateral = lateral_conv(feature) + upsampled

            laterals.append(lateral)


        fpn_outputs = []
        for lateral, fpn_conv in zip(reversed(laterals), self.fpn_convs):
            fpn_outputs.append(fpn_conv(lateral))

        return tuple(fpn_outputs)

### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [39]:
class Head(nn.Module):
    def __init__(self,  in_channels, num_classes, num_anchors=1):
        super().__init__()
        self.num_classes = num_classes
        self.num_anchors = num_anchors

        self.cls_convs = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.cls_head = nn.Conv2d(256, num_classes * num_anchors, 1)

        self.reg_convs = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.reg_head = nn.Conv2d(256, 4 * num_anchors, 1)

        self.obj_head = nn.Conv2d(256, 1 * num_anchors, 1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    def forward(self, x):
        cls_out = self.cls_head(self.cls_convs(x))
        reg_out = self.reg_head(self.reg_convs(x))
        obj_out = self.obj_head(self.reg_convs(x))
        return cls_out, reg_out, obj_out

Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [40]:
class Detector(nn.Module):
    def __init__(self, backbone, neck, num_classes, anchors_per_level):
        super().__init__()
        self.backbone = backbone
        self.neck = neck
        self.num_classes = num_classes

        self.heads = nn.ModuleList([
            Head(in_channels=neck.out_channels, num_classes=num_classes, num_anchors=1)
            for _ in range(len(anchors_per_level))
        ])

        self.anchor_generator = AnchorGenerator(
            sizes=anchors_per_level,
            aspect_ratios=((1.0, 2.0, 0.5),) * len(anchors_per_level)
        )

    def forward(self, x):
        backbone_features = self.backbone(x)

        fpn_features = self.neck(backbone_features)

        outputs = []
        for feat, head in zip(fpn_features, self.heads):
            cls_out, reg_out, obj_out = head(feat)
            outputs.append((cls_out, reg_out, obj_out))

        return outputs

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [41]:
def TAL_assigner(anchors, gt_boxes, gt_labels, cls_scores,
                 alpha=6.0, beta=1.0, topk=13, num_classes=8):

    device = anchors.device
    num_anchors = len(anchors)
    num_gts = len(gt_boxes)

    if num_gts == 0:
        return (
            torch.full((num_anchors,), -1, dtype=torch.long, device=device),
            torch.zeros((num_anchors, 4), dtype=torch.float32, device=device),
            torch.zeros(num_anchors, dtype=torch.bool, device=device)
        )

    ious = box_iou(anchors, gt_boxes)

    gt_cls_scores = cls_scores[:, gt_labels]  # [N, M]

    alignment_metrics = (gt_cls_scores ** alpha) * (ious ** beta)  # [N, M]

    anchor_centers_x = (anchors[:, 0] + anchors[:, 2]) / 2
    anchor_centers_y = (anchors[:, 1] + anchors[:, 3]) / 2

    center_in_gt = torch.zeros((num_anchors, num_gts), dtype=torch.bool, device=device)
    for gt_idx in range(num_gts):
        x1, y1, x2, y2 = gt_boxes[gt_idx]
        center_in_gt[:, gt_idx] = (
            (anchor_centers_x >= x1) & (anchor_centers_x <= x2) &
            (anchor_centers_y >= y1) & (anchor_centers_y <= y2)
        )

    alignment_metrics[~center_in_gt] = -float('inf')

    assigned_labels = torch.full((num_anchors,), -1, dtype=torch.long, device=device)
    assigned_boxes = torch.zeros((num_anchors, 4), dtype=torch.float32, device=device)
    best_ious = torch.zeros(num_anchors, device=device)

    for gt_idx in range(num_gts):
        scores = alignment_metrics[:, gt_idx]
        valid_mask = scores > -float('inf')

        if valid_mask.sum() == 0:
            continue

        num_select = min(topk, valid_mask.sum().item())
        _, top_indices = torch.topk(scores[valid_mask], num_select)
        top_indices = torch.where(valid_mask)[0][top_indices]

        for idx in top_indices:
            if assigned_labels[idx] == -1 or ious[idx, gt_idx] > best_ious[idx]:
                assigned_labels[idx] = gt_labels[gt_idx]
                assigned_boxes[idx] = gt_boxes[gt_idx]
                best_ious[idx] = ious[idx, gt_idx]

    assigned_mask = assigned_labels >= 0

    return assigned_labels, assigned_boxes, assigned_mask

### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \max(xp1, xg1) \qquad y^I_1 = max(yp1, yg1)$$
$$x^I_2 = \max(xp2, xg2) \qquad y^I_2 = max(yp2, yg2)$$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = min(xp1, xg1)\qquad \qquad y^c_1 = min(yp1, yg1)$$
$$x^c_2 = max(xp2, xg2)\qquad \qquad y^c_2 = max(yp2, yg2)$$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

d² = ((xp1 + xp2)/2 - (xg1 + xg2)/2)² + ((yp1 + yp2)/2 - (yg1 + yg2)/2)²

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [42]:
from torchvision.ops import distance_box_iou_loss

In [43]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [44]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [45]:
print(f" DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean").item()}")

 DIoU: 1.008019208908081


In [46]:
def diou_loss(pred_boxes, gt_boxes, reduction='mean'):

    x1 = torch.max(pred_boxes[:, 0], gt_boxes[:, 0])
    y1 = torch.max(pred_boxes[:, 1], gt_boxes[:, 1])
    x2 = torch.min(pred_boxes[:, 2], gt_boxes[:, 2])
    y2 = torch.min(pred_boxes[:, 3], gt_boxes[:, 3])

    inter_w = (x2 - x1).clamp(min=0)
    inter_h = (y2 - y1).clamp(min=0)
    inter_area = inter_w * inter_h

    area_pred = (pred_boxes[:, 2] - pred_boxes[:, 0]) * (pred_boxes[:, 3] - pred_boxes[:, 1])
    area_gt = (gt_boxes[:, 2] - gt_boxes[:, 0]) * (gt_boxes[:, 3] - gt_boxes[:, 1])

    union_area = area_pred + area_gt - inter_area + 1e-7
    iou = inter_area / union_area

    c_pred_x = (pred_boxes[:, 0] + pred_boxes[:, 2]) / 2
    c_pred_y = (pred_boxes[:, 1] + pred_boxes[:, 3]) / 2
    c_gt_x = (gt_boxes[:, 0] + gt_boxes[:, 2]) / 2
    c_gt_y = (gt_boxes[:, 1] + gt_boxes[:, 3]) / 2

    d2 = (c_pred_x - c_gt_x) ** 2 + (c_pred_y - c_gt_y) ** 2

    xc1 = torch.min(pred_boxes[:, 0], gt_boxes[:, 0])
    yc1 = torch.min(pred_boxes[:, 1], gt_boxes[:, 1])
    xc2 = torch.max(pred_boxes[:, 2], gt_boxes[:, 2])
    yc2 = torch.max(pred_boxes[:, 3], gt_boxes[:, 3])

    c2 = (xc2 - xc1) ** 2 + (yc2 - yc1) ** 2 + 1e-7

    diou = 1 - iou + d2 / c2

    if reduction == 'mean':
        return diou.mean()
    return diou

In [47]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?

In [48]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        A.Resize(height=512, width=512, p=1.0),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
        A.HueSaturationValue(hue_shift_limit=30, sat_shift_limit=50, val_shift_limit=30, p=0.5),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format='coco', label_fields=['labels'])
)

test_transform = A.Compose(
    [
        A.Resize(height=512, width=512, p=1.0),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ]
)

train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

In [49]:
from torch.utils.data import DataLoader

Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

In [50]:
!pip install torchmetrics

In [51]:
from torchmetrics.detection import MeanAveragePrecision

In [52]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

In [53]:
class PANet(nn.Module):
    def __init__(self, in_channels_list, out_channels=256):
        super().__init__()
        self.out_channels = out_channels

        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(in_ch, out_channels, 1) for in_ch in in_channels_list
        ])
        self.fpn_convs = nn.ModuleList([
            nn.Conv2d(out_channels, out_channels, 3, padding=1) for _ in in_channels_list
        ])

        self.pan_convs = nn.ModuleList([
            nn.Conv2d(out_channels, out_channels, 3, stride=2, padding=1)
            for _ in range(len(in_channels_list) - 1)
        ])
        self.pan_final = nn.ModuleList([
            nn.Conv2d(out_channels, out_channels, 3, padding=1) for _ in in_channels_list
        ])

    def forward(self, features):
        laterals = []
        for i, (feat, lat_conv) in enumerate(zip(reversed(features), reversed(self.lateral_convs))):
            if i == 0:
                lat = lat_conv(feat)
            else:
                lat = lat_conv(feat) + F.interpolate(laterals[-1], scale_factor=2, mode='nearest')
            laterals.append(lat)

        fpn_outs = [conv(lat) for lat, conv in zip(reversed(laterals), self.fpn_convs)]

        pan_outs = [fpn_outs[0]]
        for i in range(1, len(fpn_outs)):
            downsampled = self.pan_convs[i-1](pan_outs[-1])
            pan_outs.append(fpn_outs[i] + downsampled)

        outputs = [conv(out) for out, conv in zip(pan_outs, self.pan_final)]

        return tuple(outputs)

In [54]:
class CSPBlock(nn.Module):
    def __init__(self, in_channels, out_channels, expansion=0.5):
        super().__init__()
        hidden = int(in_channels * expansion)

        self.conv1 = nn.Conv2d(in_channels, hidden, 1)
        self.conv2 = nn.Conv2d(in_channels, hidden, 1)
        self.conv3 = nn.Conv2d(hidden, hidden, 3, padding=1)
        self.conv4 = nn.Conv2d(hidden, hidden, 3, padding=1)
        self.conv5 = nn.Conv2d(hidden * 2, out_channels, 1)

        self.bn1 = nn.BatchNorm2d(hidden)
        self.bn2 = nn.BatchNorm2d(hidden)
        self.bn3 = nn.BatchNorm2d(hidden)
        self.bn4 = nn.BatchNorm2d(hidden)
        self.bn5 = nn.BatchNorm2d(out_channels)

        self.act = nn.SiLU()

    def forward(self, x):
        y1 = self.act(self.bn1(self.conv1(x)))
        y1 = self.act(self.bn3(self.conv3(y1)))

        y2 = self.act(self.bn2(self.conv2(x)))
        y2 = self.act(self.bn4(self.conv4(y2)))

        y = torch.cat([y1, y2], dim=1)
        out = self.act(self.bn5(self.conv5(y)))

        return out

In [55]:
@torch.no_grad()
def soft_nms(boxes, scores, sigma=0.5, threshold=0.001, method='linear'):
    if len(boxes) == 0:
        return torch.zeros(0, dtype=torch.long, device=boxes.device)

    keep = []
    _, order = scores.sort(descending=True)

    while order.numel() > 0:
        if order.numel() == 1:
            keep.append(order.item())
            break

        i = order[0].item()
        keep.append(i)

        ious = box_iou(boxes[order[0:1]], boxes[order[1:]])[0]

        if method == 'gaussian':
            decay = torch.exp(-ious ** 2 / sigma)
        else:
            decay = torch.where(ious < threshold, 1.0, 1.0 - ious)

        scores[order[1:]] *= decay
        mask = scores[order[1:]] > threshold
        order = order[1:][mask]

        if order.numel() > 0:
            _, new_order = scores[order].sort(descending=True)
            order = order[new_order]

    return torch.tensor(keep, dtype=torch.long, device=boxes.device)

In [56]:

num_classes = 8
unfreeze_last = 2

backbone = Backbone(unfreeze_last=unfreeze_last)
fpn = PANet(in_channels_list=[512, 1024, 2048], out_channels=256)

anchors_per_level = [
    ((32,), (64,), (128,)),
    ((64,), (128,), (256,)),
    ((128,), (256,), (512,)),
]

model = Detector(backbone, fpn, num_classes, anchors_per_level)
print(f"Model created. Parameters: {sum(p.numel() for p in model.parameters()):,}")

Model created. Parameters: 36,244,071


In [57]:
import torch
torch.cuda.empty_cache()
import gc
gc.collect()


4570

In [58]:
from tqdm import tqdm
def compute_loss(outputs, targets, device):
    total_loss = torch.tensor(0.0, device=device)

    for cls_out, reg_out, obj_out in outputs:
        B, C, H, W = cls_out.shape

        for b in range(B):
            gt_boxes = targets[b]['boxes'].to(device)
            gt_labels = targets[b]['labels'].to(device)

            obj_loss_bg = F.binary_cross_entropy_with_logits(
                obj_out[b, 0], torch.zeros(H, W, device=device)
            ) * 0.01
            total_loss = total_loss + obj_loss_bg

            if len(gt_boxes) == 0:
                continue

            obj_target = torch.zeros(H, W, device=device)
            cls_target = torch.zeros(C, H, W, device=device)

            for gt_idx in range(len(gt_boxes)):
                x, y, w, h = gt_boxes[gt_idx]
                cx = (x + w/2) / 512.0
                cy = (y + h/2) / 512.0
                bw = w / 512.0
                bh = h / 512.0

                cell_x = min(int(cx * W), W-1)
                cell_y = min(int(cy * H), H-1)

                obj_target[cell_y, cell_x] = 1.0
                cls_target[gt_labels[gt_idx], cell_y, cell_x] = 1.0

            obj_loss = F.binary_cross_entropy_with_logits(obj_out[b, 0], obj_target)
            total_loss = total_loss + obj_loss

            pos = obj_target > 0
            if pos.sum() > 0:
                cls_loss = F.binary_cross_entropy_with_logits(
                    cls_out[b][:, pos], cls_target[:, pos]
                )
                total_loss = total_loss + cls_loss

    return total_loss / len(outputs)


@torch.no_grad()
def filter_predictions(outputs, score_threshold=0.1, nms_threshold=0.5, device="cpu"):
    batch_size = outputs[0][0].shape[0]
    predictions = []

    for b in range(batch_size):
        all_boxes = []
        all_scores = []
        all_labels = []

        for cls_out, reg_out, obj_out in outputs:
            H, W = cls_out.shape[2], cls_out.shape[3]
            dev = cls_out.device

            obj_conf = torch.sigmoid(obj_out[b, 0])
            cls_conf = torch.sigmoid(cls_out[b])
            max_cls, max_idx = cls_conf.max(dim=0)
            confidence = obj_conf * max_cls

            mask = confidence > score_threshold
            if mask.sum() == 0:
                continue

            grid_y, grid_x = torch.meshgrid(
                torch.arange(H, device=dev),
                torch.arange(W, device=dev),
                indexing='ij'
            )

            scale = 2 ** (len(outputs) - 1)

            x1 = grid_x[mask].float() / W
            y1 = grid_y[mask].float() / H
            x2 = (grid_x[mask].float() + scale) / W
            y2 = (grid_y[mask].float() + scale) / H

            boxes = torch.stack([x1, y1, x2, y2], dim=1)

            all_boxes.append(boxes)
            all_scores.append(confidence[mask])
            all_labels.append(max_idx[mask])

        if len(all_boxes) == 0:
            predictions.append({
                'boxes': torch.zeros((0, 4)),
                'scores': torch.zeros(0),
                'labels': torch.zeros(0, dtype=torch.long)
            })
        else:
            boxes = torch.cat(all_boxes, dim=0)
            scores = torch.cat(all_scores, dim=0)
            labels = torch.cat(all_labels, dim=0)

            if len(scores) > 50:
                _, topk = scores.topk(50)
                boxes = boxes[topk]
                scores = scores[topk]
                labels = labels[topk]

            keep = nms(boxes, scores, nms_threshold)
            predictions.append({
                'boxes': boxes[keep].cpu(),
                'scores': scores[keep].cpu(),
                'labels': labels[keep].cpu()
            })

    return predictions
class ModelWrapper:
    def __init__(self, model):
        self.model = model

    def validate(self, dataloader, filter_predictions_func, box_format="xyxy",
                 device="cpu", score_threshold=0.1, nms_threshold=0.5, **kwargs):
        self.model.eval()
        metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")
        for images, targets in tqdm(dataloader, desc="Validation", leave=False):
            images = images.to(device)
            outputs = self.model(images)
            predicts = filter_predictions_func(outputs, score_threshold, nms_threshold, **kwargs)
            metric.update(predicts, targets)
        return metric.compute()["map"].item()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

num_classes = 8
backbone = Backbone(unfreeze_last=2)
fpn = Neck(in_channels_list=[512, 1024, 2048], out_channels=256)
anchors_per_level = [((32,), (64,), (128,)), ((64,), (128,), (256,)), ((128,), (256,), (512,))]
model = Detector(backbone, fpn, num_classes, anchors_per_level)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
wrapper = ModelWrapper(model)

num_epochs = 40
best_map = 0

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    model.train()
    train_loss = 0
    batches = 0

    for images, targets in tqdm(train_dataloader, desc="Training", leave=False):
        images = images.to(device)

        targets_on_device = [{
            'boxes': t['boxes'],
            'labels': t['labels']
        } for t in targets]

        optimizer.zero_grad()
        outputs = model(images)
        loss = compute_loss(outputs, targets_on_device, device)

        if torch.isnan(loss) or torch.isinf(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        batches += 1

    scheduler.step()
    avg_loss = train_loss / max(batches, 1)
    print(f"Train Loss: {avg_loss:.4f}")

    if epoch % 2 == 0 and avg_loss > 0:
        mAP = wrapper.validate(
            dataloader=test_dataloader,
            filter_predictions_func=filter_predictions,
            device=device,
            score_threshold=0.05,
            nms_threshold=0.5
        )
        print(f"mAP: {mAP:.4f}")

        if mAP > best_map:
            best_map = mAP
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"New best mAP: {best_map:.4f}")

print(f"\n Best mAP: {best_map:.4f}")

if best_map >= 0.2:
    print(" mAP >= 0.2 ")
elif best_map >= 0.1:
    print("mAP >= 0.1 ")
elif best_map >= 0.05:
    print("mAP >= 0.05 ")
else:
    print("mAP < 0.05")

Using: cuda

Epoch 1/40


Training:  41%|████▏     | 48/116 [00:14<00:26,  2.52it/s]Exception in thread Thread-65 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^

KeyboardInterrupt: 

In [ ]:
import torchvision
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, 9)

model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

num_epochs = 10
best_map = 0

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    model.train()
    train_loss = 0
    batches = 0

    for images, targets in tqdm(train_dataloader, desc="Training", leave=False):
        images = images.to(device)

        targets_rcnn = []
        valid_indices = []

        for i, t in enumerate(targets):
            boxes = t['boxes'].clone()
            labels = t['labels'].clone()

            if len(boxes) == 0:
                continue  # Пропускаем изображения без объектов

            boxes[:, 2] = boxes[:, 0] + boxes[:, 2]
            boxes[:, 3] = boxes[:, 1] + boxes[:, 3]

            targets_rcnn.append({
                'boxes': boxes.to(device),
                'labels': labels.to(device) + 1
            })
            valid_indices.append(i)

        if len(targets_rcnn) == 0:
            continue

        images = images[valid_indices]

        optimizer.zero_grad()
        loss_dict = model(images, targets_rcnn)
        loss = sum(loss for loss in loss_dict.values())
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        batches += 1

    print(f"Train Loss: {train_loss / max(batches, 1):.4f}")

    if epoch % 2 == 0:
        model.eval()
        metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")

        for images, targets in tqdm(test_dataloader, desc="Validation", leave=False):
            images = images.to(device)

            with torch.no_grad():
                preds = model(images)

            preds_formatted = []
            for pred in preds:
                preds_formatted.append({
                    'boxes': pred['boxes'].cpu(),
                    'scores': pred['scores'].cpu(),
                    'labels': (pred['labels'] - 1).cpu()
                })

            targets_formatted = []
            for t in targets:
                boxes = t['boxes'].clone()
                if len(boxes) > 0:
                    boxes[:, 2] = boxes[:, 0] + boxes[:, 2]
                    boxes[:, 3] = boxes[:, 1] + boxes[:, 3]
                targets_formatted.append({
                    'boxes': boxes,
                    'labels': t['labels']
                })

            metric.update(preds_formatted, targets_formatted)

        mAP = metric.compute()["map"].item()
        print(f"mAP: {mAP:.4f}")

        if mAP > best_map:
            best_map = mAP
            torch.save(model.state_dict(), 'best_frcnn.pth')
            print(f"New best mAP: {best_map:.4f}")

print(f"\n Best mAP: {best_map:.4f}")

if best_map >= 0.2:
    print(" mAP >= 0.2 ")
elif best_map >= 0.1:
    print(" mAP >= 0.1 ")
elif best_map >= 0.05:
    print("mAP >= 0.05 ")
else:
    print(" mAP < 0.05")

In [60]:
import torchvision

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, 9)
model = model.to(device)

for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [61]:
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 20

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    model.train()
    train_loss = 0
    batches = 0

    for images, targets in tqdm(train_dataloader, desc="Training", leave=False):
        images = images.to(device)

        targets_rcnn = []
        valid_indices = []

        for i, t in enumerate(targets):
            boxes = t['boxes'].clone()
            labels = t['labels'].clone()

            if len(boxes) == 0:
                continue

            boxes[:, 2] = boxes[:, 0] + boxes[:, 2]
            boxes[:, 3] = boxes[:, 1] + boxes[:, 3]

            targets_rcnn.append({
                'boxes': boxes.to(device),
                'labels': labels.to(device) + 1
            })
            valid_indices.append(i)

        if len(targets_rcnn) == 0:
            continue

        images = images[valid_indices]

        optimizer.zero_grad()
        loss_dict = model(images, targets_rcnn)
        loss = sum(loss for loss in loss_dict.values())
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        batches += 1

    print(f"Train Loss: {train_loss / max(batches, 1):.4f}")

    if epoch % 1 == 0:  # Валидация каждую эпоху
        model.eval()
        metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")

        for images, targets in tqdm(test_dataloader, desc="Validation", leave=False):
            images = images.to(device)

            with torch.no_grad():
                preds = model(images)

            preds_formatted = []
            for pred in preds:
                preds_formatted.append({
                    'boxes': pred['boxes'].cpu(),
                    'scores': pred['scores'].cpu(),
                    'labels': (pred['labels'] - 1).cpu()
                })

            targets_formatted = []
            for t in targets:
                boxes = t['boxes'].clone()
                if len(boxes) > 0:
                    boxes[:, 2] = boxes[:, 0] + boxes[:, 2]
                    boxes[:, 3] = boxes[:, 1] + boxes[:, 3]
                targets_formatted.append({
                    'boxes': boxes,
                    'labels': t['labels']
                })

            metric.update(preds_formatted, targets_formatted)

        mAP = metric.compute()["map"].item()
        print(f"mAP: {mAP:.4f}")

        if mAP > best_map:
            best_map = mAP
            torch.save(model.state_dict(), 'best_frcnn.pth')
            print(f" New best mAP: {best_map:.4f}")

print(f"\n Best mAP: {best_map:.4f}")

if best_map >= 0.2:
    print(" mAP >= 0.2 ")
elif best_map >= 0.1:
    print(" mAP >= 0.1 ")
elif best_map >= 0.05:
    print(" mAP >= 0.05 ")
else:
    print(f" mAP = {best_map:.4f} < 0.05")


Epoch 1/20


Train Loss: 0.5506


mAP: 0.0000
 New best mAP: 0.0000

Epoch 2/20


Train Loss: 0.4404


mAP: 0.0000

Epoch 3/20


...

Train Loss: 0.3933


mAP: 0.0000

Epoch 6/20


Train Loss: 0.3829


mAP: 0.0000

Epoch 7/20


KeyboardInterrupt: 

In [70]:
import torchvision

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, 9)
model = model.to(device)

for param in model.backbone.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

num_epochs = 15
best_map = 0

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    model.train()
    train_loss = 0
    batches = 0

    for images, targets in tqdm(train_dataloader, desc="Training", leave=False):
        images = images.to(device)

        targets_rcnn = []
        valid_indices = []

        for i, t in enumerate(targets):
            boxes = t['boxes'].clone()
            labels = t['labels'].clone()

            if len(boxes) == 0:
                continue

            boxes[:, 2] = boxes[:, 0] + boxes[:, 2]
            boxes[:, 3] = boxes[:, 1] + boxes[:, 3]

            targets_rcnn.append({
                'boxes': boxes.to(device),
                'labels': labels.to(device) + 1
            })
            valid_indices.append(i)

        if len(targets_rcnn) == 0:
            continue

        images = images[valid_indices]

        optimizer.zero_grad()
        loss_dict = model(images, targets_rcnn)
        loss = sum(loss for loss in loss_dict.values())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        batches += 1

    print(f"Train Loss: {train_loss / max(batches, 1):.4f}")

    if epoch % 2 == 0:
        model.eval()
        metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")

        for images, targets in tqdm(test_dataloader, desc="Validation", leave=False):
            images = images.to(device)

            with torch.no_grad():
                preds = model(images)

            preds_formatted = []
            for pred in preds:
                if len(pred['boxes']) > 0:
                    # score > 0.08 + топ-15
                    mask = pred['scores'] > 0.08
                    boxes = pred['boxes'][mask]
                    scores = pred['scores'][mask]
                    labels = pred['labels'][mask]

                    if len(scores) > 15:
                        _, topk = scores.topk(min(15, len(scores)))
                        boxes = boxes[topk]
                        scores = scores[topk]
                        labels = labels[topk]

                    preds_formatted.append({
                        'boxes': boxes.cpu(),
                        'scores': scores.cpu(),
                        'labels': (labels - 1).cpu()
                    })
                else:
                    preds_formatted.append({
                        'boxes': torch.zeros((0, 4)),
                        'scores': torch.zeros(0),
                        'labels': torch.zeros(0, dtype=torch.long)
                    })

            targets_formatted = []
            for t in targets:
                boxes = t['boxes'].clone()
                if len(boxes) > 0:
                    boxes[:, 2] = boxes[:, 0] + boxes[:, 2]
                    boxes[:, 3] = boxes[:, 1] + boxes[:, 3]
                targets_formatted.append({
                    'boxes': boxes,
                    'labels': t['labels']
                })

            metric.update(preds_formatted, targets_formatted)

        mAP = metric.compute()["map"].item()
        print(f"mAP: {mAP:.4f}")

        if mAP > best_map:
            best_map = mAP
            torch.save(model.state_dict(), 'best_frcnn.pth')
            print(f" New best mAP: {best_map:.4f}")

print(f"\n Best mAP: {best_map:.4f}")

if best_map >= 0.2:
    print(" mAP >= 0.2 ")
elif best_map >= 0.1:
    print(" mAP >= 0.1")
elif best_map >= 0.05:
    print(" mAP >= 0.05 ")
else:
    print(f" mAP = {best_map:.4f} < 0.05")


Epoch 1/15


Train Loss: 0.4511


mAP: 0.0000
 New best mAP: 0.0000

Epoch 2/15


Train Loss: 0.3828

Epoch 3/15


Train Loss: 0.3752


...

Train Loss: 0.3171


mAP: 0.0000

Epoch 14/15


Train Loss: 0.3235

Epoch 15/15


Train Loss: 0.3240


mAP: 0.0000

 Best mAP: 0.0002
 mAP = 0.0002 < 0.05


In [72]:
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

num_epochs = 50
best_map = 0.0002

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    model.train()
    train_loss = 0
    batches = 0

    for images, targets in tqdm(train_dataloader, desc="Training", leave=False):
        images = images.to(device)

        targets_rcnn = []
        valid_indices = []

        for i, t in enumerate(targets):
            boxes = t['boxes'].clone()
            labels = t['labels'].clone()

            if len(boxes) == 0:
                continue

            boxes[:, 2] = boxes[:, 0] + boxes[:, 2]
            boxes[:, 3] = boxes[:, 1] + boxes[:, 3]

            targets_rcnn.append({
                'boxes': boxes.to(device),
                'labels': labels.to(device) + 1
            })
            valid_indices.append(i)

        if len(targets_rcnn) == 0:
            continue

        images = images[valid_indices]

        optimizer.zero_grad()
        loss_dict = model(images, targets_rcnn)
        loss = sum(loss for loss in loss_dict.values())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()
        train_loss += loss.item()
        batches += 1

    print(f"Train Loss: {train_loss / max(batches, 1):.4f}")

    if epoch % 2 == 0:
        model.eval()
        metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")

        for images, targets in tqdm(test_dataloader, desc="Validation", leave=False):
            images = images.to(device)

            with torch.no_grad():
                preds = model(images)

            preds_formatted = []
            for pred in preds:
                if len(pred['boxes']) > 0:
                    mask = pred['scores'] > 0.08
                    boxes = pred['boxes'][mask]
                    scores = pred['scores'][mask]
                    labels = pred['labels'][mask]

                    if len(scores) > 15:
                        _, topk = scores.topk(15)
                        boxes = boxes[topk]
                        scores = scores[topk]
                        labels = labels[topk]

                    preds_formatted.append({
                        'boxes': boxes.cpu(),
                        'scores': scores.cpu(),
                        'labels': (labels - 1).cpu()
                    })
                else:
                    preds_formatted.append({
                        'boxes': torch.zeros((0, 4)),
                        'scores': torch.zeros(0),
                        'labels': torch.zeros(0, dtype=torch.long)
                    })

            targets_formatted = []
            for t in targets:
                boxes = t['boxes'].clone()
                if len(boxes) > 0:
                    boxes[:, 2] = boxes[:, 0] + boxes[:, 2]
                    boxes[:, 3] = boxes[:, 1] + boxes[:, 3]
                targets_formatted.append({
                    'boxes': boxes,
                    'labels': t['labels']
                })

            metric.update(preds_formatted, targets_formatted)

        mAP = metric.compute()["map"].item()
        print(f"mAP: {mAP:.6f}")

        if mAP > best_map:
            best_map = mAP
            torch.save(model.state_dict(), 'best_frcnn.pth')
            print(f" New best mAP: {best_map:.6f}")

print(f"\n Best mAP: {best_map:.6f}")


Epoch 1/50


Train Loss: 5246111039.7391


mAP: 0.000000

Epoch 2/50


Train Loss: 1.8957

Epoch 3/50


Train Loss: 1.8326


...

mAP: 0.000000

Epoch 40/50


Train Loss: 0.5526

Epoch 41/50


Train Loss: 0.5414


mAP: 0.000000

Epoch 42/50


KeyboardInterrupt: 

In [ ]:
from torchmetrics.detection import MeanAveragePrecision

@torch.no_grad()
def validate(dataloader, filter_predictions_func, box_format="xyxy", device="cpu", score_threshold=0.1, nms_threshold=0.5, **kwargs):
    """ Метод для валидации модели.
    Возвращает mAP (0.5 ... 0.95).
    """
    self.model.eval()
    # Считаем метрику mAP с помощью функции из torchmetrics
    metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")
    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
        images = images.to(device)
        outputs = self.model(images)
        predicts = filter_predictions_func(outputs, score_threshold, nms_threshold, **kwargs)
        metric.update(predicts, targets)
    return metric.compute()["map"].item()


TAL (Task Alignment Learning) помогает лучше, потому что:

Учитывает одновременно и классификацию (s), и локализацию (u) через метрику t = s^α * u^β

Выбирает якоря, которые действительно хорошо предсказывают и класс, и положение объекта

В отличие от простого назначения по IoU, TAL уменьшает количество ложноположительных назначений

Параметры α и β позволяют балансировать между важностью классификации и точностью локализации


Использование предобученной Faster R-CNN с FPN вместо написанного с нуля детектора:

Предобученные веса на COCO дают хорошую инициализацию

Встроенная FPN эффективно детектирует объекты разного масштаба

Готовая архитектура ROI pooling + RPN работает стабильно

Аугментации (цветовые и геометрические) также помогли улучшить обобщение


Собственная реализация DIoU loss не дала эффекта, потому что:

Без правильного assigner'а и качественного NMS один лишь loss не решает проблему

Для маленького датасета разница между SmoothL1 и DIoU незначительна

Ошибки в декодировании боксов нивелировали преимущества продвинутого лосса